# NetShield — Phase 6: Robust Evaluation & Error Analysis

### Purpose

This notebook is the next stage after `05_modeling_experiments.ipynb`.

It does **not** introduce random new models or chase higher scores. Its purpose is to determine whether the very high CICIDS2017 within-dataset result is robust, investigate duplicate/capture-scenario leakage, and explain the asymmetric cross-dataset failures.

The notebook follows the NetShield research question:

> **How dataset-dependent are ML-based network intrusion detectors, and what roles do feature representation, traffic distribution, and class imbalance play in cross-dataset generalization?**

### Important methodological principles

- Raw data are never overwritten.
- `source_file` is metadata, never a predictive feature.
- Preprocessing statistics are learned from training data only.
- Meaningful zeros are preserved.
- `±inf` is converted to missing values.
- Clearly non-negative traffic features with invalid negative values are treated as missing; `Init_Win_bytes_forward/backward` are not blanket-cleaned because `-1` can be a sentinel.
- Exact/feature duplicates are diagnosed before deciding how they affect evaluation.
- The original random CICIDS split is retained as a **baseline**, not silently discarded.
- Source-aware and duplicate-aware evaluations are treated as robustness/sensitivity analyses.
- Cross-dataset transfer continues to use the verified 11-feature semantic representation.
- The destination dataset is never used to tune the source-trained model.

## Runtime note

CICIDS2017 contains about 2.83 million rows. The previous notebook demonstrated that full native CICIDS training is expensive on a laptop.

Therefore this notebook has two clearly separated modes:

- **SCREENING (default):** uses a fixed, stratified training cap for the expensive robustness model fits. This lets you complete the analysis reliably.
- **FULL:** set `RUN_MODE = "full"` only if you want the complete-data robustness fit and are prepared for substantially longer runtime.

The split construction and all evaluation logic remain the same in both modes.

In [ ]:
from pathlib import Path
import json
import warnings
import hashlib
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, average_precision_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

# -----------------------------
# Runtime control
# -----------------------------
RUN_MODE = "screening"   # "screening" or "full"

# In screening mode, this is the maximum number of CICIDS training rows
# used for each robustness model fit. The test set remains untouched.
SCREENING_TRAIN_CAP = 600_000

# Keep model families consistent with Phase 5.
# Fewer trees/iterations here are intentional for the robustness sensitivity
# analysis and are recorded in the manifest.
RF_TREES = 100
HGB_ITER = 100

print("RUN_MODE:", RUN_MODE)
print("SCREENING_TRAIN_CAP:", SCREENING_TRAIN_CAP)

## 1. Locate the NetShield project

The notebook first tries to find the project root automatically. It expects:

```text
NetShield/
├── data/raw/UNSW-NB15/
├── data/raw/CICIDS2017/
└── outputs/
```

If the project cannot be found, the notebook stops rather than silently using a different dataset.

In [ ]:
from pathlib import Path

def find_project_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/mnt/data"),
        Path.home() / "NetShield",
    ]
    for p in candidates:
        if (p / "data" / "raw" / "UNSW-NB15").exists() and (p / "data" / "raw" / "CICIDS2017").exists():
            return p
    return None

PROJECT_ROOT = find_project_root()

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "NetShield project root was not found. Open/run this notebook from the NetShield "
        "project folder containing data/raw/UNSW-NB15 and data/raw/CICIDS2017."
    )

UNSW_ROOT = PROJECT_ROOT / "data" / "raw" / "UNSW-NB15"
CICIDS_ROOT = PROJECT_ROOT / "data" / "raw" / "CICIDS2017"

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "robust_evaluation"
TABLE_DIR = OUTPUT_ROOT / "tables"
PLOT_DIR = OUTPUT_ROOT / "plots"
for p in [TABLE_DIR, PLOT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Output root :", OUTPUT_ROOT)

## 2. Load UNSW-NB15 and CICIDS2017

UNSW uses the official training/testing files.

CICIDS is loaded from all CSV files, retaining `source_file` only as metadata. We do not use it as a predictive feature.

In [ ]:
# UNSW
unsw_train_path = UNSW_ROOT / "Training and Testing Sets" / "UNSW_NB15_training-set.csv"
unsw_test_path = UNSW_ROOT / "Training and Testing Sets" / "UNSW_NB15_testing-set.csv"

if not unsw_train_path.exists() or not unsw_test_path.exists():
    raise FileNotFoundError("Official UNSW training/testing files were not found.")

unsw_train = pd.read_csv(unsw_train_path, low_memory=False)
unsw_test = pd.read_csv(unsw_test_path, low_memory=False)

# CICIDS
cicids_files = sorted(CICIDS_ROOT.rglob("*.csv"))
if not cicids_files:
    raise FileNotFoundError(f"No CICIDS CSV files found under {CICIDS_ROOT}")

parts = []
for path in cicids_files:
    part = pd.read_csv(path, low_memory=False)
    part.columns = part.columns.str.strip()
    part["source_file"] = path.name
    parts.append(part)

cicids = pd.concat(parts, ignore_index=True)

print("UNSW train:", unsw_train.shape)
print("UNSW test :", unsw_test.shape)
print("CICIDS    :", cicids.shape)
print("CICIDS source files:", cicids["source_file"].nunique())

## 3. Create binary labels without copying the entire CICIDS dataframe

We avoid `cicids.copy()` because CICIDS is large. Only the new label column is added.

In [ ]:
unsw_train["label_binary"] = pd.to_numeric(
    unsw_train["label"], errors="coerce"
).astype("int8")

unsw_test["label_binary"] = pd.to_numeric(
    unsw_test["label"], errors="coerce"
).astype("int8")

cicids["label_binary"] = (
    cicids["Label"].astype(str).str.strip().str.upper().ne("BENIGN")
).astype("int8")

print("UNSW train:")
display(unsw_train["label_binary"].value_counts(normalize=True).sort_index().rename("proportion"))

print("UNSW test:")
display(unsw_test["label_binary"].value_counts(normalize=True).sort_index().rename("proportion"))

print("CICIDS:")
display(cicids["label_binary"].value_counts(normalize=True).sort_index().rename("proportion"))

## 4. Define native predictive features and consistent cleaning

Identifiers and metadata are excluded.

For CICIDS, clearly non-negative traffic quantities are checked for invalid negative values. We deliberately do **not** treat every negative number as invalid because `Init_Win_bytes_forward` and `Init_Win_bytes_backward` can contain sentinel values such as `-1`.

The cleaning is performed before model fitting, while imputation remains inside the training pipeline so medians are learned from training data only.

In [ ]:
UNSW_DROP = {"id", "label", "label_binary", "attack_cat"}

CICIDS_DROP = {
    "Label", "label_binary", "source_file",
    "Flow ID", "Source IP", "Destination IP", "Timestamp"
}

unsw_native_features = [
    c for c in unsw_train.columns
    if c not in UNSW_DROP and pd.api.types.is_numeric_dtype(unsw_train[c])
]

cicids_native_features = [
    c for c in cicids.columns
    if c not in CICIDS_DROP and pd.api.types.is_numeric_dtype(cicids[c])
]

# Features that should not legitimately be negative.
CICIDS_NONNEGATIVE = [
    "Flow Duration", "Total Fwd Packets", "Total Backward Packets",
    "Total Length of Fwd Packets", "Total Length of Bwd Packets",
    "Fwd Packet Length Max", "Fwd Packet Length Min",
    "Fwd Packet Length Mean", "Fwd Packet Length Std",
    "Bwd Packet Length Max", "Bwd Packet Length Min",
    "Bwd Packet Length Mean", "Bwd Packet Length Std",
    "Flow Bytes/s", "Flow Packets/s",
    "Flow IAT Mean", "Flow IAT Std", "Flow IAT Max", "Flow IAT Min",
    "Fwd IAT Total", "Fwd IAT Mean", "Fwd IAT Std", "Fwd IAT Max", "Fwd IAT Min",
    "Bwd IAT Total", "Bwd IAT Mean", "Bwd IAT Std", "Bwd IAT Max", "Bwd IAT Min",
    "Fwd PSH Flags", "Bwd PSH Flags", "Fwd URG Flags", "Bwd URG Flags",
    "Fwd Header Length", "Bwd Header Length",
    "Fwd Packets/s", "Bwd Packets/s",
    "Min Packet Length", "Max Packet Length", "Packet Length Mean",
    "Packet Length Std", "Packet Length Variance",
    "Average Packet Size", "Avg Fwd Segment Size", "Avg Bwd Segment Size",
    "Subflow Fwd Packets", "Subflow Fwd Bytes", "Subflow Bwd Packets", "Subflow Bwd Bytes",
    "act_data_pkt_fwd", "min_seg_size_forward",
    "Active Mean", "Active Std", "Active Max", "Active Min",
    "Idle Mean", "Idle Std", "Idle Max", "Idle Min"
]

# Keep only columns actually present.
CICIDS_NONNEGATIVE = [c for c in CICIDS_NONNEGATIVE if c in cicids.columns]

# Replace infinities first.
for df, cols in [
    (unsw_train, unsw_native_features),
    (unsw_test, unsw_native_features),
    (cicids, cicids_native_features),
]:
    df[cols] = df[cols].replace([np.inf, -np.inf], np.nan)

# Convert clearly invalid negative values to NaN in CICIDS.
cicids_invalid_counts = {}
for c in CICIDS_NONNEGATIVE:
    mask = pd.to_numeric(cicids[c], errors="coerce") < 0
    n = int(mask.sum())
    if n:
        cicids.loc[mask, c] = np.nan
    cicids_invalid_counts[c] = n

# Remove constant CICIDS features based on the complete cleaned dataset.
constant_cicids = [
    c for c in cicids_native_features
    if cicids[c].nunique(dropna=False) <= 1
]

cicids_native_features_clean = [
    c for c in cicids_native_features if c not in constant_cicids
]

pd.DataFrame({
    "feature": list(cicids_invalid_counts.keys()),
    "negative_values_replaced": list(cicids_invalid_counts.values())
}).to_csv(TABLE_DIR / "01_cicids_invalid_value_cleaning.csv", index=False)

pd.DataFrame({"constant_feature": constant_cicids}).to_csv(
    TABLE_DIR / "02_cicids_constant_features_removed.csv", index=False
)

print("UNSW native features:", len(unsw_native_features))
print("CICIDS native features before constant removal:", len(cicids_native_features))
print("CICIDS constant features removed:", len(constant_cicids))
print("CICIDS native features used:", len(cicids_native_features_clean))

## 5. Duplicate diagnostics — do not delete blindly

There are several different meanings of "duplicate":

1. exact full-row duplicates,
2. identical predictive feature vectors,
3. identical predictive features with conflicting labels.

The third category is especially important. If the same feature vector occurs with both benign and attack labels, it cannot simply be treated as harmless duplication.

We save diagnostics before deciding how the robust split should behave.

In [ ]:
# Work only with the cleaned native predictive features plus binary label.
cicids_eval = cicids.dropna(subset=["label_binary"])

# Exact duplicate rows in the original dataframe are already known from EDA,
# but recompute a compact diagnostic using predictive features + label.
dup_feature_label = cicids_eval[cicids_native_features_clean + ["label_binary"]].duplicated(keep=False)

# Hashing a row of predictive features gives a compact grouping key.
feature_hash = pd.util.hash_pandas_object(
    cicids_eval[cicids_native_features_clean],
    index=False
)

label_by_hash = pd.DataFrame({
    "feature_hash": feature_hash.to_numpy(),
    "label_binary": cicids_eval["label_binary"].to_numpy()
})

group_summary = (
    label_by_hash.groupby("feature_hash")["label_binary"]
    .agg(["size", "nunique"])
    .rename(columns={"size": "rows_in_feature_group", "nunique": "unique_labels"})
)

n_feature_duplicate_rows = int(
    group_summary.loc[group_summary["rows_in_feature_group"] > 1, "rows_in_feature_group"].sum()
)
n_conflicting_groups = int((group_summary["unique_labels"] > 1).sum())
n_conflicting_rows = int(
    group_summary.loc[group_summary["unique_labels"] > 1, "rows_in_feature_group"].sum()
)

duplicate_summary = pd.DataFrame([{
    "cicids_rows": len(cicids_eval),
    "exact_feature_label_duplicate_rows": int(dup_feature_label.sum()),
    "feature_duplicate_rows": n_feature_duplicate_rows,
    "conflicting_feature_groups": n_conflicting_groups,
    "rows_in_conflicting_feature_groups": n_conflicting_rows
}])

display(duplicate_summary)
duplicate_summary.to_csv(TABLE_DIR / "03_duplicate_summary.csv", index=False)

# Keep the hash as metadata for grouping; never as a predictive feature.
cicids_eval = cicids_eval.copy()
cicids_eval["_feature_group_hash"] = feature_hash.to_numpy()

## 5A. Quantify potential leakage in the random split

Before fitting models, we explicitly measure whether identical predictive feature groups cross the train/test boundary.

This is crucial: if many identical feature vectors occur on both sides of the random split, a very high score can be partly explained by duplicated observations rather than genuine generalization.

In [ ]:
def crossing_group_count(train_idx, test_idx):
    assignment = pd.Series("train", index=np.arange(len(cicids_eval)))
    assignment.iloc[test_idx] = "test"
    tmp = pd.DataFrame({
        "feature_hash": cicids_eval["_feature_group_hash"].to_numpy(),
        "partition": assignment.to_numpy()
    })
    parts_per_group = tmp.groupby("feature_hash")["partition"].nunique()
    return int((parts_per_group > 1).sum())

leakage_diag = pd.DataFrame([
    {
        "split": "random_stratified",
        "feature_groups_crossing_train_test": crossing_group_count(
            random_train_idx, random_test_idx
        )
    },
    {
        "split": "duplicate_aware",
        "feature_groups_crossing_train_test": crossing_group_count(
            duplicate_train_idx, duplicate_test_idx
        )
    },
    {
        "split": "capture_aware",
        "feature_groups_crossing_train_test": crossing_group_count(
            capture_train_idx, capture_test_idx
        )
    }
])

display(leakage_diag)
leakage_diag.to_csv(TABLE_DIR / "04b_duplicate_group_crossing_diagnostic.csv", index=False)

## 6. Build three CICIDS evaluation splits

### A. Random stratified split
This reproduces the logic of Phase 5 and is retained as the baseline.

### B. Duplicate-aware split
Feature-identical flows are kept in the same partition, preventing identical predictive vectors from crossing train/test.

### C. Capture-aware split
Entire source files are held out together. This is a robustness experiment, not a replacement for the random baseline, because CICIDS source files have very different attack compositions.

The capture-aware split below deliberately chooses a holdout with both benign and attack traffic:
- test: Friday DDoS + Friday PortScan + Monday
- train: all remaining source files

This asks a concrete question:

> How well does the model generalize when the major DDoS/PortScan capture scenarios are unseen during training, while benign traffic from Monday remains in the test set?

In [ ]:
from sklearn.model_selection import train_test_split

# -----------------------------
# A. Random stratified split
# -----------------------------
idx = np.arange(len(cicids_eval))
y_all = cicids_eval["label_binary"].to_numpy()

random_train_idx, random_test_idx = train_test_split(
    idx,
    test_size=0.20,
    stratify=y_all,
    random_state=RANDOM_STATE
)

# -----------------------------
# B. Duplicate-aware split
# -----------------------------
# GroupShuffleSplit ensures each feature-identical group stays together.
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
dup_train_pos, dup_test_pos = next(
    gss.split(
        np.zeros(len(cicids_eval)),
        y_all,
        groups=cicids_eval["_feature_group_hash"].to_numpy()
    )
)

duplicate_train_idx = idx[dup_train_pos]
duplicate_test_idx = idx[dup_test_pos]

# -----------------------------
# C. Capture-aware split
# -----------------------------
capture_test_files = {
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Monday-WorkingHours.pcap_ISCX.csv",
}

source_series = cicids_eval["source_file"].astype(str)

capture_test_mask = source_series.isin(capture_test_files).to_numpy()
capture_train_idx = idx[~capture_test_mask]
capture_test_idx = idx[capture_test_mask]

def split_summary(name, train_idx, test_idx):
    return {
        "split": name,
        "train_rows": len(train_idx),
        "test_rows": len(test_idx),
        "train_attack_rate": float(y_all[train_idx].mean()),
        "test_attack_rate": float(y_all[test_idx].mean()),
        "test_source_files": int(cicids_eval.iloc[test_idx]["source_file"].nunique())
    }

split_table = pd.DataFrame([
    split_summary("random_stratified", random_train_idx, random_test_idx),
    split_summary("duplicate_aware", duplicate_train_idx, duplicate_test_idx),
    split_summary("capture_aware", capture_train_idx, capture_test_idx),
])

display(split_table)
split_table.to_csv(TABLE_DIR / "04_cicids_split_comparison.csv", index=False)

print("Capture-aware test files:")
display(cicids_eval.iloc[capture_test_idx]["source_file"].value_counts())

## 7. Prevent an unfairly huge laptop workload

The robustness analysis should compare splits, not merely spend hours rebuilding the same enormous forest.

In **screening mode**, the training data are reduced to a fixed stratified cap *after* the split has been constructed. The test data remain untouched.

This does not replace the original Phase-5 results. It is a controlled robustness/sensitivity analysis.

Set `RUN_MODE = "full"` above if you intentionally want to fit on all available training rows.

In [ ]:
def stratified_cap_indices(indices, labels, cap, random_state=42):
    if cap is None or len(indices) <= cap:
        return np.asarray(indices)

    y = np.asarray(labels)[indices]
    selected, _ = train_test_split(
        np.asarray(indices),
        test_size=(len(indices) - cap) / len(indices),
        stratify=y,
        random_state=random_state
    )
    return np.asarray(selected)

def maybe_cap(train_idx):
    if RUN_MODE == "full":
        return np.asarray(train_idx)
    return stratified_cap_indices(
        train_idx,
        y_all,
        SCREENING_TRAIN_CAP,
        RANDOM_STATE
    )

split_indices = {
    "random_stratified": (maybe_cap(random_train_idx), random_test_idx),
    "duplicate_aware": (maybe_cap(duplicate_train_idx), duplicate_test_idx),
    "capture_aware": (maybe_cap(capture_train_idx), capture_test_idx),
}

split_runtime_table = pd.DataFrame([
    {
        "split": name,
        "original_train_rows": len({
            "random_stratified": random_train_idx,
            "duplicate_aware": duplicate_train_idx,
            "capture_aware": capture_train_idx
        }[name]),
        "model_train_rows": len(tr),
        "test_rows": len(te)
    }
    for name, (tr, te) in split_indices.items()
])

display(split_runtime_table)
split_runtime_table.to_csv(TABLE_DIR / "05_runtime_training_sizes.csv", index=False)

## 8. Define the four model families

The same four model families are retained:

- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting (implemented as HistGradientBoostingClassifier)

For this robustness notebook, Random Forest and Gradient Boosting use fewer trees/iterations than Phase 5 to keep the sensitivity analysis computationally practical. This is recorded explicitly and must not be presented as a direct hyperparameter-for-hyperparameter replacement of Phase 5.

In [ ]:
robust_models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE
        ))
    ]),
    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(
            random_state=RANDOM_STATE
        ))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=RF_TREES,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),
    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingClassifier(
            max_iter=HGB_ITER,
            random_state=RANDOM_STATE
        ))
    ])
}

print(list(robust_models.keys()))

## 9. Evaluation helper

Besides the standard metrics, we save:

- TP, TN, FP, FN
- FPR
- FNR
- predicted attack rate

The predicted attack rate is useful for explaining the asymmetric transfer behavior observed in Phase 5.

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test,
                   experiment, model_name, split_name):
    fitted = clone(model)
    fitted.fit(X_train, y_train)

    y_pred = fitted.predict(X_test)

    if hasattr(fitted, "predict_proba"):
        y_score = fitted.predict_proba(X_test)[:, 1]
    elif hasattr(fitted, "decision_function"):
        y_score = fitted.decision_function(X_test)
    else:
        y_score = None

    tn, fp, fn, tp = confusion_matrix(
        y_test, y_pred, labels=[0, 1]
    ).ravel()

    return {
        "experiment": experiment,
        "split": split_name,
        "model": model_name,
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "pr_auc": (
            average_precision_score(y_test, y_score)
            if y_score is not None else np.nan
        ),
        "roc_auc": (
            roc_auc_score(y_test, y_score)
            if y_score is not None and len(np.unique(y_test)) == 2
            else np.nan
        ),
        "fpr": fp / (fp + tn) if (fp + tn) else np.nan,
        "fnr": fn / (fn + tp) if (fn + tp) else np.nan,
        "predicted_attack_rate": float(np.mean(y_pred)),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)
    }, fitted, y_pred, y_score

## 10. CICIDS native robustness experiment

We now compare:

- random stratified
- duplicate-aware
- capture-aware

using the same native CICIDS feature representation.

This is the key validation of the earlier 0.997-ish CICIDS result.

In [ ]:
cicids_native_X = cicids_eval[cicids_native_features_clean]
cicids_native_y = cicids_eval["label_binary"].astype("int8")

native_rows = []
native_fitted = {}
native_predictions = {}

for split_name, (train_idx, test_idx) in split_indices.items():
    X_train = cicids_native_X.iloc[train_idx]
    y_train = cicids_native_y.iloc[train_idx]
    X_test = cicids_native_X.iloc[test_idx]
    y_test = cicids_native_y.iloc[test_idx]

    print(f"\n=== {split_name} ===")
    print("train:", X_train.shape, "test:", X_test.shape)

    for model_name, model in robust_models.items():
        print(f"  fitting {model_name} ...")
        row, fitted, y_pred, y_score = evaluate_model(
            model, X_train, y_train, X_test, y_test,
            "CICIDS_native_robustness", model_name, split_name
        )
        native_rows.append(row)
        native_fitted[(split_name, model_name)] = fitted
        native_predictions[(split_name, model_name)] = {
            "test_idx": test_idx,
            "y_true": y_test.to_numpy(),
            "y_pred": y_pred,
            "y_score": y_score
        }

native_robust_results = pd.DataFrame(native_rows)
display(
    native_robust_results[
        ["split","model","precision","recall","f1","macro_f1","pr_auc","fpr","fnr",
         "predicted_attack_rate","tn","fp","fn","tp"]
    ].sort_values(["split","f1"], ascending=[True, False])
)

native_robust_results.to_csv(
    TABLE_DIR / "06_cicids_native_robustness_results.csv", index=False
)

## 11. Confusion matrices for the robustness experiment

These plots make the difference between ordinary random-split performance and leakage-aware performance visible.

In [ ]:
for (split_name, model_name), pred_info in native_predictions.items():
    cm = confusion_matrix(pred_info["y_true"], pred_info["y_pred"], labels=[0,1])
    fig, ax = plt.subplots(figsize=(5,4))
    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Benign", "Attack"]
    ).plot(ax=ax, values_format="d", colorbar=False)
    ax.set_title(f"CICIDS — {split_name} — {model_name}")
    fig.tight_layout()
    safe = re.sub(r"[^A-Za-z0-9_-]+", "_", f"{split_name}_{model_name}")
    fig.savefig(PLOT_DIR / f"cm_{safe}.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

## 12. Compare the robustness results with the Phase-5 results

If Phase 5 result CSVs exist in the same project output tree, this cell loads them automatically.

The comparison is diagnostic:

> Does the extremely high random-split score remain high when duplicates or capture scenarios are controlled?

In [ ]:
# Search for Phase 5 result files without assuming an exact folder name.
candidate_paths = list(PROJECT_ROOT.glob("outputs/**/02_cicids_native_within_results.csv"))

phase5_cicids = None
if candidate_paths:
    phase5_cicids = pd.read_csv(candidate_paths[0])
    phase5_cicids["source"] = "Phase5_random_split"
    print("Loaded:", candidate_paths[0])
    display(phase5_cicids)
else:
    print("Phase-5 CICIDS result CSV not found under outputs/. The current robustness results are still valid.")

if phase5_cicids is not None:
    phase5_view = phase5_cicids[["model","f1","macro_f1","pr_auc","fpr"]].copy()
    phase5_view["split"] = "phase5_random_baseline"

    robust_view = native_robust_results[
        ["split","model","f1","macro_f1","pr_auc","fpr"]
    ].copy()

    comparison = pd.concat([phase5_view, robust_view], ignore_index=True)
    display(comparison.sort_values(["model","split"]))
    comparison.to_csv(
        TABLE_DIR / "07_phase5_vs_robust_cicids_comparison.csv", index=False
    )

## 13. Common semantic representation

The same verified 11-feature representation from Phase 5 is used for cross-dataset error analysis.

Important: this notebook does not reinterpret the semantic mappings.

In [ ]:
COMMON_FEATURES = [
    "flow_duration_sec", "fwd_packets", "bwd_packets",
    "fwd_bytes", "bwd_bytes", "total_packets", "total_bytes",
    "packet_rate", "fwd_packet_rate", "bwd_packet_rate",
    "avg_packet_size"
]

def safe_divide_series(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce")
    if np.isscalar(denominator):
        denominator = pd.Series(denominator, index=numerator.index)
    else:
        denominator = pd.to_numeric(denominator, errors="coerce")
    denominator = denominator.replace(0, np.nan)
    return numerator.div(denominator)

def build_unsw_common(df):
    out = pd.DataFrame(index=df.index)
    out["flow_duration_sec"] = pd.to_numeric(df["dur"], errors="coerce")
    out["fwd_packets"] = pd.to_numeric(df["spkts"], errors="coerce")
    out["bwd_packets"] = pd.to_numeric(df["dpkts"], errors="coerce")
    out["fwd_bytes"] = pd.to_numeric(df["sbytes"], errors="coerce")
    out["bwd_bytes"] = pd.to_numeric(df["dbytes"], errors="coerce")
    out["total_packets"] = out["fwd_packets"] + out["bwd_packets"]
    out["total_bytes"] = out["fwd_bytes"] + out["bwd_bytes"]
    out["packet_rate"] = safe_divide_series(out["total_packets"], out["flow_duration_sec"])
    out["fwd_packet_rate"] = safe_divide_series(out["fwd_packets"], out["flow_duration_sec"])
    out["bwd_packet_rate"] = safe_divide_series(out["bwd_packets"], out["flow_duration_sec"])
    out["avg_packet_size"] = safe_divide_series(out["total_bytes"], out["total_packets"])
    out["label_binary"] = df["label_binary"].astype("int8")
    return out

def build_cicids_common(df):
    out = pd.DataFrame(index=df.index)
    out["flow_duration_sec"] = safe_divide_series(df["Flow Duration"], 1_000_000)
    out["fwd_packets"] = pd.to_numeric(df["Total Fwd Packets"], errors="coerce")
    out["bwd_packets"] = pd.to_numeric(df["Total Backward Packets"], errors="coerce")
    out["fwd_bytes"] = pd.to_numeric(df["Total Length of Fwd Packets"], errors="coerce")
    out["bwd_bytes"] = pd.to_numeric(df["Total Length of Bwd Packets"], errors="coerce")
    out["total_packets"] = out["fwd_packets"] + out["bwd_packets"]
    out["total_bytes"] = out["fwd_bytes"] + out["bwd_bytes"]
    out["packet_rate"] = pd.to_numeric(df["Flow Packets/s"], errors="coerce")
    out["fwd_packet_rate"] = pd.to_numeric(df["Fwd Packets/s"], errors="coerce")
    out["bwd_packet_rate"] = pd.to_numeric(df["Bwd Packets/s"], errors="coerce")
    out["avg_packet_size"] = pd.to_numeric(df["Average Packet Size"], errors="coerce")
    out["label_binary"] = df["label_binary"].astype("int8")
    return out

unsw_common_train = build_unsw_common(unsw_train)
unsw_common_test = build_unsw_common(unsw_test)
cicids_common = build_cicids_common(cicids_eval)

print(unsw_common_train.shape, unsw_common_test.shape, cicids_common.shape)

## 14. Cross-dataset error analysis

This stage does not try to improve the cross-dataset score. It asks **what kind of errors are being made**.

We fit the four models on the source dataset's common semantic features and evaluate on the destination dataset.

For the CICIDS source, a fixed stratified source-training cap is used in screening mode to keep runtime manageable. The destination dataset is untouched.

In [ ]:
common_models = robust_models

X_unsw_common_train = unsw_common_train[COMMON_FEATURES].replace([np.inf, -np.inf], np.nan)
y_unsw_common_train = unsw_common_train["label_binary"]

X_unsw_common_test = unsw_common_test[COMMON_FEATURES].replace([np.inf, -np.inf], np.nan)
y_unsw_common_test = unsw_common_test["label_binary"]

X_cicids_common = cicids_common[COMMON_FEATURES].replace([np.inf, -np.inf], np.nan)
y_cicids_common = cicids_common["label_binary"]

# Optional cap for CICIDS as a source dataset.
cicids_source_idx = np.arange(len(X_cicids_common))
cicids_source_train_idx = stratified_cap_indices(
    cicids_source_idx,
    y_cicids_common.to_numpy(),
    SCREENING_TRAIN_CAP if RUN_MODE == "screening" else None,
    RANDOM_STATE
)

cross_rows = []
cross_predictions = {}
cross_fitted = {}

for model_name, model in common_models.items():
    print(f"UNSW -> CICIDS: {model_name}")
    row, fitted, y_pred, y_score = evaluate_model(
        model,
        X_unsw_common_train, y_unsw_common_train,
        X_cicids_common, y_cicids_common,
        "UNSW_to_CICIDS_common_error_analysis", model_name, "common_semantic"
    )
    cross_rows.append(row)
    cross_predictions[("UNSW_to_CICIDS", model_name)] = {
        "y_true": y_cicids_common.to_numpy(),
        "y_pred": y_pred,
        "y_score": y_score
    }
    cross_fitted[("UNSW_to_CICIDS", model_name)] = fitted

for model_name, model in common_models.items():
    print(f"CICIDS -> UNSW: {model_name}")
    row, fitted, y_pred, y_score = evaluate_model(
        model,
        X_cicids_common.iloc[cicids_source_train_idx],
        y_cicids_common.iloc[cicids_source_train_idx],
        X_unsw_common_test, y_unsw_common_test,
        "CICIDS_to_UNSW_common_error_analysis", model_name, "common_semantic"
    )
    cross_rows.append(row)
    cross_predictions[("CICIDS_to_UNSW", model_name)] = {
        "y_true": y_unsw_common_test.to_numpy(),
        "y_pred": y_pred,
        "y_score": y_score
    }
    cross_fitted[("CICIDS_to_UNSW", model_name)] = fitted

cross_error_results = pd.DataFrame(cross_rows)
display(
    cross_error_results[
        ["experiment","model","precision","recall","f1","macro_f1",
         "pr_auc","fpr","fnr","predicted_attack_rate","tn","fp","fn","tp"]
    ].sort_values(["experiment","f1"], ascending=[True, False])
)

cross_error_results.to_csv(
    TABLE_DIR / "08_cross_dataset_error_analysis_results.csv", index=False
)

## 15. Cross-dataset confusion matrices

In [ ]:
for (direction, model_name), pred_info in cross_predictions.items():
    cm = confusion_matrix(pred_info["y_true"], pred_info["y_pred"], labels=[0,1])
    fig, ax = plt.subplots(figsize=(5,4))
    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Benign", "Attack"]
    ).plot(ax=ax, values_format="d", colorbar=False)
    ax.set_title(f"{direction} — {model_name}")
    fig.tight_layout()
    safe = re.sub(r"[^A-Za-z0-9_-]+", "_", f"{direction}_{model_name}")
    fig.savefig(PLOT_DIR / f"cm_cross_{safe}.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

## 16. Feature importance: what does each dataset model learn?

We inspect tree-based feature importance for the native CICIDS robustness models.

For the cross-dataset experiment, we inspect feature importance in the common semantic representation.

This is descriptive: importance does not prove causality.

In [ ]:
importance_rows = []

def extract_feature_importance(fitted_pipeline):
    model = fitted_pipeline.named_steps["model"]
    if hasattr(model, "feature_importances_"):
        return model.feature_importances_
    return None

# Native CICIDS importance
for (split_name, model_name), fitted in native_fitted.items():
    if model_name in {"Decision Tree", "Random Forest", "Gradient Boosting"}:
        values = extract_feature_importance(fitted)
        if values is not None:
            for feature, importance in zip(cicids_native_features_clean, values):
                importance_rows.append({
                    "experiment": "CICIDS_native_robustness",
                    "split": split_name,
                    "model": model_name,
                    "feature": feature,
                    "importance": float(importance)
                })

# Common cross-dataset importance
for (direction, model_name), fitted in cross_fitted.items():
    if model_name in {"Decision Tree", "Random Forest", "Gradient Boosting"}:
        values = extract_feature_importance(fitted)
        if values is not None:
            for feature, importance in zip(COMMON_FEATURES, values):
                importance_rows.append({
                    "experiment": direction,
                    "split": "common_semantic",
                    "model": model_name,
                    "feature": feature,
                    "importance": float(importance)
                })

importance_df = pd.DataFrame(importance_rows)
importance_df.to_csv(TABLE_DIR / "09_feature_importance.csv", index=False)

for (experiment, split_name, model_name), g in importance_df.groupby(
    ["experiment","split","model"]
):
    top = g.sort_values("importance", ascending=False).head(15)
    print(f"\n{experiment} | {split_name} | {model_name}")
    display(top[["feature","importance"]])

## 17. Distribution-shift diagnostics on the common semantic features

We calculate a practical set of distribution diagnostics between UNSW and CICIDS:

- median
- IQR
- zero rate
- standardized mean difference using robust scales

The goal is not to declare a feature "bad". The goal is to quantify how different the two environments are.

In [ ]:
def robust_scale_iqr(s):
    q1, q3 = np.nanpercentile(s, [25, 75])
    iqr = q3 - q1
    return iqr if iqr > 0 else np.nan

shift_rows = []

for feature in COMMON_FEATURES:
    a = pd.to_numeric(unsw_common_train[feature], errors="coerce").replace([np.inf,-np.inf], np.nan)
    b = pd.to_numeric(cicids_common[feature], errors="coerce").replace([np.inf,-np.inf], np.nan)

    med_a, med_b = np.nanmedian(a), np.nanmedian(b)
    iqr_a, iqr_b = robust_scale_iqr(a), robust_scale_iqr(b)

    pooled_scale = np.nanmedian([iqr_a, iqr_b])
    robust_median_shift = abs(med_a - med_b) / pooled_scale if pooled_scale and np.isfinite(pooled_scale) else np.nan

    shift_rows.append({
        "feature": feature,
        "unsw_median": med_a,
        "cicids_median": med_b,
        "unsw_iqr": iqr_a,
        "cicids_iqr": iqr_b,
        "unsw_zero_rate": float(np.mean(a == 0)),
        "cicids_zero_rate": float(np.mean(b == 0)),
        "absolute_zero_rate_difference": abs(float(np.mean(a == 0)) - float(np.mean(b == 0))),
        "robust_median_shift": robust_median_shift
    })

shift_df = pd.DataFrame(shift_rows).sort_values(
    "robust_median_shift", ascending=False
)

display(shift_df)
shift_df.to_csv(TABLE_DIR / "10_common_feature_distribution_shift.csv", index=False)

## 18. Relate distribution shift to cross-dataset performance

This is exploratory, not a causal test.

We summarize the cross-dataset F1 loss from Phase 5 if those results are available, then compare it with the magnitude of semantic feature shifts.

The final interpretation should say "associated with" rather than "caused by".

In [ ]:
phase5_files = {
    "unsw_to_cicids": list(PROJECT_ROOT.glob("outputs/**/03_unsw_to_cicids_common_results.csv")),
    "cicids_to_unsw": list(PROJECT_ROOT.glob("outputs/**/04_cicids_to_unsw_common_results.csv")),
    "weighted": list(PROJECT_ROOT.glob("outputs/**/07_class_weighted_transfer_results.csv")),
}

for k, paths in phase5_files.items():
    print(k, ":", paths[0] if paths else "not found")

# Simple shift ranking summary for reporting.
shift_summary = shift_df[
    ["feature","robust_median_shift","absolute_zero_rate_difference"]
].copy()

shift_summary["shift_rank"] = np.arange(1, len(shift_summary)+1)
display(shift_summary)

## 19. Save a compact research manifest

This records exactly what this notebook did, including the screening/full mode and computational settings.

In [ ]:
manifest = {
    "project": "NetShield",
    "phase": "06_robust_evaluation_and_error_analysis",
    "random_state": RANDOM_STATE,
    "run_mode": RUN_MODE,
    "screening_train_cap": SCREENING_TRAIN_CAP,
    "rf_trees": RF_TREES,
    "hist_gradient_boosting_iterations": HGB_ITER,
    "primary_task": "binary intrusion detection",
    "primary_metrics": [
        "precision", "recall", "f1", "macro_f1",
        "pr_auc", "fpr", "fnr", "confusion_matrix"
    ],
    "cicids_splits": [
        "random_stratified",
        "duplicate_aware",
        "capture_aware"
    ],
    "capture_test_files": sorted(capture_test_files),
    "native_feature_rule": "identifiers and metadata excluded; constant CICIDS features removed",
    "source_file_used_as_predictor": False,
    "duplicate_feature_hash_used_only_for_grouping": True,
    "cross_dataset_representation": COMMON_FEATURES,
    "next_analysis": [
        "compare random vs duplicate-aware vs capture-aware CICIDS performance",
        "interpret asymmetric cross-dataset errors",
        "compare feature importance across datasets",
        "connect distribution shift with transfer degradation",
        "perform only justified preprocessing sensitivity experiments"
    ]
}

with open(OUTPUT_ROOT / "phase6_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Saved:", OUTPUT_ROOT / "phase6_manifest.json")
print("\nPhase 6 complete.")